In [1]:
import os
import sys

# Limpiar variables problemáticas
for var in [
    "MASTER",
    "SPARK_MASTER",
    "PYSPARK_SUBMIT_ARGS",
    "SPARK_REMOTE",
    "SPARK_CONNECT_MODE_ENABLED"
]:
    os.environ.pop(var, None)

# Forzar el Python correcto
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("Busqueda_MapReduce")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
    .getOrCreate()
)

sc = spark.sparkContext
print("master:", sc.master)
print("python:", sys.executable)
print(sc.parallelize([1, 2, 3]).map(lambda x: x + 1).collect())

master: local[1]
python: C:\Users\Theas\AppData\Local\Programs\Python\Python311\python.exe
[2, 3, 4]


In [2]:
# Lista de documentos (nombre_archivo, contenido)
documents = [
    ("doc1.txt", "Snoopy es un personaje del comic Peanuts creado por Charles Schulz"),
    ("doc1.txt", "Snoopy a veces imagina que es un piloto volando sobre Belgica"),
    ("doc2.txt", "Snoopy disfruta aprender programacion y ciencia de datos"),
    ("doc2.txt", "Snoopy podria usar Spark para analizar informacion"),
    ("doc3.txt", "Snoopy organiza a los Peanuts como si fueran un cluster distribuido"),
    ("doc3.txt", "MapReduce fue inspirado en programacion funcional segun Snoopy"),
    ("doc4.txt", "Snoopy ama ver Pokemon con Woodstock los fines de semana"),
    ("doc4.txt", "Muchos entrenadores Pokemon quisieran tener a Snoopy en su equipo"),
    ("doc5.txt", "Snoopy viajo a Belgica y probo el famoso chocolate belga"),
    ("doc5.txt", "Bruselas recibio a Snoopy con un desfile especial"),
    ("doc6.txt", "Algunas convenciones de Snoopy y Pokemon se han realizado en Belgica")
]

# Convertir a RDD
docs_rdd = sc.parallelize(documents)

# Mostrar datos
docs_rdd.collect()

[('doc1.txt',
  'Snoopy es un personaje del comic Peanuts creado por Charles Schulz'),
 ('doc1.txt', 'Snoopy a veces imagina que es un piloto volando sobre Belgica'),
 ('doc2.txt', 'Snoopy disfruta aprender programacion y ciencia de datos'),
 ('doc2.txt', 'Snoopy podria usar Spark para analizar informacion'),
 ('doc3.txt',
  'Snoopy organiza a los Peanuts como si fueran un cluster distribuido'),
 ('doc3.txt',
  'MapReduce fue inspirado en programacion funcional segun Snoopy'),
 ('doc4.txt', 'Snoopy ama ver Pokemon con Woodstock los fines de semana'),
 ('doc4.txt',
  'Muchos entrenadores Pokemon quisieran tener a Snoopy en su equipo'),
 ('doc5.txt', 'Snoopy viajo a Belgica y probo el famoso chocolate belga'),
 ('doc5.txt', 'Bruselas recibio a Snoopy con un desfile especial'),
 ('doc6.txt',
  'Algunas convenciones de Snoopy y Pokemon se han realizado en Belgica')]

In [3]:
pattern = "Snoopy"
print("Patrón:", pattern)

Patrón: Snoopy


In [4]:
mapped = docs_rdd.flatMap(
    lambda x: [x[0]] if pattern.lower() in x[1].lower() else []
)

print(mapped.collect())

['doc1.txt', 'doc1.txt', 'doc2.txt', 'doc2.txt', 'doc3.txt', 'doc3.txt', 'doc4.txt', 'doc4.txt', 'doc5.txt', 'doc5.txt', 'doc6.txt']


In [5]:
result = mapped.distinct()
print("Documentos que contienen el patrón:")
print(result.collect())

Documentos que contienen el patrón:
['doc1.txt', 'doc2.txt', 'doc3.txt', 'doc4.txt', 'doc5.txt', 'doc6.txt']


In [6]:
mapped_count = docs_rdd.map(
    lambda x: (x[0], 1) if pattern.lower() in x[1].lower() else None
).filter(lambda x: x is not None)

count_result = mapped_count.reduceByKey(lambda a, b: a + b)

print("Ocurrencias por documento:")
print(count_result.collect())

Ocurrencias por documento:
[('doc1.txt', 2), ('doc2.txt', 2), ('doc3.txt', 2), ('doc4.txt', 2), ('doc5.txt', 2), ('doc6.txt', 1)]
